# Data Pipeline Functionality

Step-by-step walkthrough of the full protein ESP pipeline.

Each section shows the **viability** of that step for every protein
(whether prerequisites exist, whether outputs are already present),
optionally runs the step, and displays the resulting metadata/statistics.

Set `RUN_PIPELINE = True` in the configuration cell to execute steps;
leave it `False` to inspect status only.

In [ ]:
# ── Display setup ─────────────────────────────────────────────────────────────
import os
os.environ.setdefault("DISPLAY", ":0")
os.environ.setdefault("WAYLAND_DISPLAY", "wayland-0")
os.environ.setdefault("XDG_RUNTIME_DIR", "/mnt/wslg/runtime-dir")

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import sys, time, json
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path("../").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import get_config, get_data_root
from src.utils.paths import ProteinPaths
from src.utils.io import load_metadata
from src.structure.af_api import (
    download_protein, find_downloaded_protein_ids, read_uniprot_ids
)
from src.electrostatics.run_pdb2pqr import process_pdb2pqr
from src.electrostatics.run_apbs import process_apbs
from src.surface.mesh import build_mesh
from src.surface.esp_mapping import sample_esp
from src.analysis.metrics import evaluate_protein

cfg       = get_config()
data_root = get_data_root()
pd.set_option("display.max_colwidth", 40)
pd.set_option("display.float_format", "{:.4f}".format)
print("data_root:", data_root)

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# UniProt accession IDs to process.
# Edit this list directly, or load from a file:
#   UNIPROT_IDS = read_uniprot_ids(Path("../data/test_ids.txt"))
UNIPROT_IDS = [
    "P01082",
    "Q16613",
    "B1WC58",
]

# Set True to actually run each step; False for status inspection only.
RUN_PIPELINE = True

print(f"UniProt IDs : {UNIPROT_IDS}")
print(f"Run pipeline: {RUN_PIPELINE}")
print(f"data_root   : {data_root}")

---
## Pipeline Status Overview

Status key: `done` — output exists | `ready` — prereqs met, will run | `miss` — prereqs missing

In [ ]:
# ── Pipeline status overview ──────────────────────────────────────────────────
# Resolves any already-downloaded proteins and shows per-step status.

def _step_status(prereq_ok: bool, output_ok: bool) -> str:
    if output_ok:   return "done"
    if prereq_ok:   return "ready"
    return "miss"

rows = []
for uid in UNIPROT_IDS:
    pids = find_downloaded_protein_ids(uid, data_root)
    if not pids:
        rows.append({
            "uniprot_id": uid, "protein_id": "(not downloaded)",
            "download": "ready",
            "pdb2pqr": "miss", "apbs": "miss",
            "mesh": "miss", "esp": "miss", "evaluate": "miss",
        })
        continue
    for pid in pids:
        p = ProteinPaths(pid, data_root)
        rows.append({
            "uniprot_id": uid,
            "protein_id": pid,
            "download" : "done",
            "pdb2pqr"  : _step_status(p.pdb_path.exists(),     p.pqr_path.exists()),
            "apbs"     : _step_status(p.pqr_path.exists(),     p.dx_path.exists()),
            "mesh"     : _step_status(p.pqr_path.exists(),     p.pqr_mesh_path.exists()),
            "esp"      : _step_status(p.pqr_mesh_path.exists() and p.dx_path.exists(),
                                      p.all_sampled_exist()),
            "evaluate" : _step_status(p.all_sampled_exist(),   p.is_evaluated()),
        })

status_df = pd.DataFrame(rows)
display(status_df)

---
## Step 1 — Download AlphaFold Structures

Downloads all fragments (F1, F2, …) per UniProt ID from the AlphaFold EBI API.
Skips UniProt IDs where any fragment is already present on disk.

In [ ]:
# ── Step 1: Download ──────────────────────────────────────────────────────────
protein_ids_by_uid = {}   # uniprot_id → [protein_id, ...]
rows = []

for uid in UNIPROT_IDS:
    existing = find_downloaded_protein_ids(uid, data_root)
    if existing:
        status = "skip"
        protein_ids_by_uid[uid] = existing
    elif RUN_PIPELINE:
        t0 = time.perf_counter()
        ok = download_protein(uid, data_root)
        elapsed = time.perf_counter() - t0
        pids = find_downloaded_protein_ids(uid, data_root) if ok else []
        protein_ids_by_uid[uid] = pids
        status = f"done ({elapsed:.1f}s)" if ok else "FAILED"
    else:
        protein_ids_by_uid[uid] = []
        status = "would run"

    pids = protein_ids_by_uid.get(uid, [])
    for pid in (pids or [uid]):
        row = {"uniprot_id": uid, "protein_id": pid, "download": status}
        if pid != uid:
            try:
                meta = load_metadata(pid, data_root)
                row["protein_name"]    = meta.get("protein_name", "")[:35]
                row["organism"]        = meta.get("organism", "")[:30]
                row["sequence_length"] = meta.get("sequence_length")
                row["fragment"]        = meta.get("fragment", 1)
                row["plddt_mean"]      = meta.get("plddt_mean")
            except Exception:
                pass
        rows.append(row)
    if not pids:
        rows.append({"uniprot_id": uid, "protein_id": "—", "download": status})

df1 = pd.DataFrame(rows)
display(df1)

---
## Step 2 — PDB2PQR

Assigns atomic charges and radii using PDB2PQR (PARSE force field, pH 7.0).
Input: `.pdb`   Output: `.pqr`, `_pdb2pqr.pdb`

In [ ]:
# ── Step 2: PDB2PQR ───────────────────────────────────────────────────────────
rows = []
for uid, pids in protein_ids_by_uid.items():
    for pid in pids:
        p = ProteinPaths(pid, data_root)
        prereq = p.pdb_path.exists()
        done   = p.pqr_path.exists()
        if done:
            action = "skip"
        elif prereq and RUN_PIPELINE:
            t0 = time.perf_counter()
            ok = process_pdb2pqr(pid, data_root)
            elapsed = time.perf_counter() - t0
            action = f"done ({elapsed:.1f}s)" if ok else "FAILED"
        elif prereq:
            action = "would run"
        else:
            action = "miss — no PDB"

        rows.append({
            "protein_id"  : pid,
            "pdb_exists"  : prereq,
            "pqr_exists"  : p.pqr_path.exists(),
            "pdb2pqr"     : action,
            "pqr_size_kb" : round(p.pqr_path.stat().st_size / 1024, 1)
                            if p.pqr_path.exists() else None,
        })

df2 = pd.DataFrame(rows)
display(df2)

---
## Step 3 — APBS Electrostatics

Computes the electrostatic potential grid using APBS.
Input: `.pqr`   Output: `.dx` ESP grid

In [ ]:
# ── Step 3: APBS ──────────────────────────────────────────────────────────────
rows = []
for uid, pids in protein_ids_by_uid.items():
    for pid in pids:
        p = ProteinPaths(pid, data_root)
        prereq = p.pqr_path.exists()
        done   = p.dx_path.exists()
        if done:
            action = "skip"
        elif prereq and RUN_PIPELINE:
            t0 = time.perf_counter()
            ok = process_apbs(pid, data_root)
            elapsed = time.perf_counter() - t0
            action = f"done ({elapsed:.1f}s)" if ok else "FAILED"
        elif prereq:
            action = "would run"
        else:
            action = "miss — no PQR"

        rows.append({
            "protein_id" : pid,
            "pqr_exists" : prereq,
            "dx_exists"  : p.dx_path.exists(),
            "apbs"       : action,
            "dx_size_mb" : round(p.dx_path.stat().st_size / 1_048_576, 2)
                           if p.dx_path.exists() else None,
        })

df3 = pd.DataFrame(rows)
display(df3)

---
## Step 4 — PQR Surface Mesh

Generates the solvent-excluded surface (SES) mesh from the PQR file using MSMS.
Input: `.pqr`   Output: `_pqr_mesh.npz` (verts, normals, faces, ses_area)

In [ ]:
# ── Step 4: PQR Mesh ──────────────────────────────────────────────────────────
rows = []
for uid, pids in protein_ids_by_uid.items():
    for pid in pids:
        p = ProteinPaths(pid, data_root)
        prereq = p.pqr_path.exists()
        done   = p.pqr_mesh_path.exists()
        if done:
            action = "skip"
        elif prereq and RUN_PIPELINE:
            t0 = time.perf_counter()
            try:
                build_mesh(p.pqr_path, pid, data_root)
                elapsed = time.perf_counter() - t0
                action = f"done ({elapsed:.1f}s)"
            except Exception as e:
                action = f"FAILED: {e}"
        elif prereq:
            action = "would run"
        else:
            action = "miss — no PQR"

        row = {"protein_id": pid, "pqr_exists": prereq,
               "mesh_exists": p.pqr_mesh_path.exists(), "mesh": action}
        if p.pqr_mesh_path.exists():
            d = np.load(p.pqr_mesh_path)
            row["n_verts"]   = int(d["verts"].shape[0])
            row["n_faces"]   = int(d["faces"].shape[0])
            row["ses_area_A2"] = round(float(d["ses_area"]), 1)
            row["vert_density"] = round(row["n_verts"] / row["ses_area_A2"], 3)
        rows.append(row)

df4 = pd.DataFrame(rows)
display(df4)

---
## Step 5 — ESP Sampling

Samples ESP from the APBS `.dx` grid onto the PQR mesh using nearest-neighbor
interpolation, then reconstructs via Laplacian with 5% curvature-prioritised
known vertices.

Input: `_pqr_mesh.npz` + `.dx`
Output: `_pqr_mesh_interp.npz`, `_pqr_mesh_laplacian.npz`

In [ ]:
# ── Step 5: ESP Sampling ──────────────────────────────────────────────────────
rows = []
for uid, pids in protein_ids_by_uid.items():
    for pid in pids:
        p = ProteinPaths(pid, data_root)
        prereq = p.pqr_mesh_path.exists() and p.dx_path.exists()
        done   = p.all_sampled_exist()
        if done:
            action = "skip"
        elif prereq and RUN_PIPELINE:
            t0 = time.perf_counter()
            ok = sample_esp(pid, data_root)
            elapsed = time.perf_counter() - t0
            action = f"done ({elapsed:.1f}s)" if ok else "FAILED"
        elif prereq:
            action = "would run"
        else:
            missing = []
            if not p.pqr_mesh_path.exists(): missing.append("mesh")
            if not p.dx_path.exists():       missing.append("dx")
            action = f"miss — no {', '.join(missing)}"

        row = {"protein_id": pid, "prereq_ok": prereq,
               "sampled": p.all_sampled_exist(), "esp_sampling": action}
        if p.pqr_interp_path.exists():
            d = np.load(p.pqr_interp_path)
            esp = d["esp_verts"]
            row["esp_min"]  = round(float(esp.min()), 3)
            row["esp_max"]  = round(float(esp.max()), 3)
            row["esp_mean"] = round(float(esp.mean()), 3)
        rows.append(row)

df5 = pd.DataFrame(rows)
display(df5)

---
## Step 6 — Evaluate

Compares Laplacian-reconstructed ESP against the interpolated reference using
Pearson r and RMSE (computed on per-face ESP values).

Input: `_pqr_mesh_interp.npz` + `_pqr_mesh_laplacian.npz`
Output: `pearson_r_pqr`, `rmse_pqr` written to metadata JSON

In [ ]:
# ── Step 6: Evaluate ──────────────────────────────────────────────────────────
rows = []
for uid, pids in protein_ids_by_uid.items():
    for pid in pids:
        p = ProteinPaths(pid, data_root)
        prereq = p.all_sampled_exist()
        done   = p.is_evaluated()
        if done:
            action = "skip"
        elif prereq and RUN_PIPELINE:
            t0 = time.perf_counter()
            try:
                evaluate_protein(pid, data_root, write_metadata=True)
                elapsed = time.perf_counter() - t0
                action = f"done ({elapsed:.1f}s)"
            except Exception as e:
                action = f"FAILED: {e}"
        elif prereq:
            action = "would run"
        else:
            action = "miss — no sampled files"

        row = {"protein_id": pid, "prereq_ok": prereq,
               "evaluated": p.is_evaluated(), "evaluate": action}
        if p.metadata_path.exists():
            try:
                meta = load_metadata(pid, data_root)
                row["pearson_r_pqr"] = meta.get("pearson_r_pqr")
                row["rmse_pqr"]      = meta.get("rmse_pqr")
            except Exception:
                pass
        rows.append(row)

df6 = pd.DataFrame(rows)
display(df6)

---
## Metadata Summary

Full accumulated metadata for all processed proteins.

In [ ]:
# ── Full metadata summary ─────────────────────────────────────────────────────
SUMMARY_COLS = [
    "protein_id", "uniprot_id", "fragment",
    "protein_name", "organism", "sequence_length",
    "plddt_mean", "plddt_median",
    "af_model_version",
    "pearson_r_pqr", "rmse_pqr",
    "time_esp_sampling_sec", "time_total_sec",
    "pipeline_complete",
]

rows = []
for uid, pids in protein_ids_by_uid.items():
    for pid in pids:
        p = ProteinPaths(pid, data_root)
        if not p.metadata_path.exists():
            rows.append({"protein_id": pid, "uniprot_id": uid})
            continue
        try:
            meta = load_metadata(pid, data_root)
            row  = {"protein_id": pid, "uniprot_id": uid}
            for col in SUMMARY_COLS[2:]:
                val = meta.get(col)
                # Truncate long strings
                if isinstance(val, str) and len(val) > 40:
                    val = val[:37] + "..."
                row[col] = val
            rows.append(row)
        except Exception as e:
            rows.append({"protein_id": pid, "uniprot_id": uid, "error": str(e)})

summary_df = pd.DataFrame(rows)[[c for c in SUMMARY_COLS if c in pd.DataFrame(rows).columns]]
pd.set_option("display.max_columns", None)
display(summary_df)